# Домашнее задание 6. Python для анализа данных

**Гипотеза:** «раньше снимали настоящее кино» — с ростом года выпуска средний рейтинг фильмов становится ниже.

Проверяем по данным `movies_stats.csv` по алгоритму из условия.

In [1]:
import pandas as pd

movies = pd.read_csv('movies_stats.csv')
print(movies.shape)
movies.head()

(100004, 6)


,userId,movieId,rating,timestamp,title,genres
0,1,31,2.5,1260759144,Dangerous Minds (1995),Drama
1,1,1029,3.0,1260759179,Dumbo (1941),Animation|Children|Drama|Musical
2,1,1061,3.0,1260759182,Sleepers (1996),Thriller
3,1,1129,2.0,1260759185,Escape from New York (1981),Action|Adventure|Sci-Fi|Thriller
4,1,1172,4.0,1260759205,Cinema Paradiso (Nuovo cinema Paradiso) (1989),Drama


### 1. Список годов с 1950 по 2010

In [2]:
years = list(range(1950, 2011))
print(years[0], years[-1], len(years))

1950 2010 61


### 2. Функция `production_year`

Проходим по всем годам из `years`; если номер года встречается в названии фильма, возвращаем его как год выпуска, иначе — 1900.

In [3]:
def production_year(title):
    for year in years:
        if str(year) in title:
            return year
    return 1900

### 3. Столбец `year`

Один фильм встречается в таблице много раз — по разу на каждую оценку. Поэтому считаем год только для уникальных названий и затем сопоставляем его всем строкам через `map`. Результат тот же, что у `movies['title'].apply(production_year)`, но функция вызывается по разу на фильм, а не на каждую оценку.

In [4]:
unique_titles = movies['title'].unique()
year_by_title = {title: production_year(title) for title in unique_titles}
movies['year'] = movies['title'].map(year_by_title)

print(f'Строк: {len(movies)}, уникальных фильмов: {len(unique_titles)}')
movies[['title', 'year']].drop_duplicates().head()

Строк: 100004, уникальных фильмов: 9064


,title,year
0,Dangerous Minds (1995),1995
1,Dumbo (1941),1900
2,Sleepers (1996),1996
3,Escape from New York (1981),1981
4,Cinema Paradiso (Nuovo cinema Paradiso) (1989),1989


In [5]:
# контроль: результат совпадает с прямым применением функции к каждой строке
(movies['year'] == movies['title'].apply(production_year)).all()

True

### 4. Средний рейтинг по годам, по убыванию

In [6]:
rating_by_year = movies.groupby('year')['rating'].mean().sort_values(ascending=False)
rating_by_year.head(10)

year
1957    4.014241
1972    4.011136
1952    4.000000
1954    3.994220
1951    3.983539
1974    3.978704
1962    3.952446
1950    3.915254
1977    3.905786
1964    3.841492
Name: rating, dtype: float64

In [7]:
rating_by_year.tail(10)

year
1999    3.471692
1992    3.466754
2001    3.460052
2002    3.456297
1990    3.449169
2005    3.448434
2003    3.444777
1996    3.426600
1997    3.415764
2000    3.356104
Name: rating, dtype: float64

### Проверка гипотезы

Год 1900 — служебное значение «год не найден», в анализ тренда его не включаем. Смотрим корреляцию года выпуска со средним рейтингом и средние по десятилетиям.

In [8]:
trend = rating_by_year.drop(1900, errors='ignore').sort_index()
correlation = pd.Series(trend.index, index=trend.index).corr(trend)
print(f'Корреляция года выпуска и среднего рейтинга: {correlation:.3f}')

Корреляция года выпуска и среднего рейтинга: -0.823


In [9]:
known_years = movies[movies['year'] != 1900]
by_decade = known_years.groupby(known_years['year'] // 10 * 10)['rating'].mean().round(3)
by_decade.index.name = 'decade'
by_decade

decade
1950    3.879
1960    3.783
1970    3.809
1980    3.607
1990    3.483
2000    3.459
2010    3.537
Name: rating, dtype: float64

## Вывод

По этим данным гипотеза подтверждается: чем позже год выпуска, тем ниже средний рейтинг.

- Корреляция года выпуска и среднего рейтинга по годам равна −0,82 — это сильная отрицательная связь.
- По десятилетиям средний рейтинг снижается с 3,88 в 1950-х до 3,46 в 2000-х. В группу «2010» попадает только один год, поэтому небольшой рост там не показателен.
- В начале отсортированного списка — 1950–1970-е годы, в конце — 1990–2000-е.

Оговорка: это не доказывает, что старое кино само по себе лучше. Старые фильмы в датасете — в основном классика, которую до сих пор смотрят и высоко оценивают (ошибка выжившего), а новые оценивают все подряд, и удачные, и проходные. Кроме того, фильмам до 1950 года и фильмам без года в названии алгоритм присваивает 1900, и в анализ тренда они не вошли.